In [ ]:
from pytket import Circuit, qasm

from pytket import Circuit

import scipy as sp
import numpy as np
from sympy import Symbol
from qibo import hamiltonians
from qibo.symbols import X, Y, Z

import matplotlib.pyplot as plt

In [ ]:
def substitute_ansatz_circuit(symbolic_circuit, params):
    assert len(params) == len(symbolic_circuit.free_symbols())
    symbol_dict = {s: p for s, p in zip(symbolic_circuit.free_symbols(), params)}
    state_prep_circuit = symbolic_circuit.copy()
    state_prep_circuit.symbol_substitution(symbol_dict)
    return state_prep_circuit

def unitary_expectation(H, U=None, ket0=None):
    if ket0 is None:
        ket0 = np.zeros((H.shape[0],), dtype=complex)
        ket0[0] = 1.0
    if U is None:
        psi = ket0
    else:
        psi = U @ ket0
    return np.vdot(psi, H @ psi).real

In [ ]:
def hdw_ansatz(nqubits, nlayers=1):
    """
    Constructs a hardware-efficient ansatz circuit for VQE using pytket with symbolic parameters.

    Args:
        nqubits (int): Number of qubits in the circuit.
        nlayers (int, optional): Number of ansatz layers to apply. Defaults to 1.

    Returns:
        tuple:
            - Circuit: A pytket Circuit object with symbolic parameters.
            - dict: A dictionary mapping parameter names to Symbol objects.
    """
    # Initialize the circuit with the specified number of qubits
    circuit = Circuit(nqubits)
    
    # Dictionary to store symbolic parameters
    theta_symbols = {}
    
    for layer in range(nlayers):
        # Apply parameterized RY and RZ gates to each qubit
        for q in range(nqubits):
            # Create unique symbol names for each gate
            ry_name = f"ry_layer{layer}_q{q}"
            rz_name = f"rz_layer{layer}_q{q}"
            
            # Create Symbol objects
            ry_theta = Symbol(ry_name)
            rz_theta = Symbol(rz_name)
            
            # Add the symbols to the dictionary
            theta_symbols[ry_name] = ry_theta
            theta_symbols[rz_name] = rz_theta
            
            # Add RY and RZ gates with symbolic parameters
            circuit.Ry(ry_theta, q).Rz(rz_theta, q)
        
        # Apply ZZ gates in a nearest-neighbor pattern for even starting pairs
        for q in range(0, nqubits - 1, 2):
            # Create symbol name
            zz_name = f"zz_layer{layer}_q{q}"
            # Create Symbol object
            zz_theta = Symbol(zz_name)
            # Add object to dictionary
            theta_symbols[zz_name] = zz_theta
            circuit.ZZPhase(zz_theta, q, q + 1)
        # Apply ZZ gates in a nearest-neighbor pattern for odd starting pairs
        for q in range(1, nqubits-1, 2):
            # Create symbol name
            zz_name = f"zz_layer{layer}_q{q}"
            # Create Symbol object
            zz_theta = Symbol(zz_name)
            # Add object to dictionary
            theta_symbols[zz_name] = zz_theta
            circuit.ZZPhase(zz_theta, q, q + 1)
        # Add ZZ to the first and last qubit
        zz_name = f"zz_layer{layer}_qboundary"
        zz_theta = Symbol(zz_name)
        theta_symbols[zz_name] = zz_theta
        circuit.ZZPhase(zz_theta, 0, nqubits-1)

    # Additional layer of RY and RZ gates (Optional?)
    for q in range(nqubits):
        final_ry_name = f"add_ry_q{q}"
        final_rz_name = f"add_rz_q{q}"
        
        final_ry_theta = Symbol(final_ry_name)
        final_rz_theta = Symbol(final_rz_name)
        
        theta_symbols[final_ry_name] = final_ry_theta
        theta_symbols[final_rz_name] = final_rz_theta
        
        circuit.Ry(final_ry_theta, q)
        circuit.Rz(final_rz_theta, q)
    
    return circuit, theta_symbols

## Train VQE

In [ ]:
nqubits = 10
delta = 0.5
# h hamiltonian
# XXZ
h_sym = sum([X(i)*X(i+1)+ Y(i)*Y(i+1) + delta* Z(i)*Z(i+1) for i in range(nqubits-1)]
            +[X(0)*X(nqubits-1)+ Y(0)*Y(nqubits-1) + delta* Z(0)*Z(nqubits-1)])
# TLFIM 2ZZ + Z + X
# h_sym = sum([2 * X(i)*X(i+1) + Z(i) + X(i) for i in range(nqubits-1)]
#             + [Z(nqubits-1)+ X(nqubits-1)])

h_qibo = hamiltonians.SymbolicHamiltonian(h_sym)
print(len(h_qibo.matrix[0]))
target_energy = np.real(np.min(np.asarray(h_qibo.eigenvalues())))
# create VQE ansatz circuit
nlayer = 2
symbolic_circuit, symbols = hdw_ansatz(nqubits, nlayer)
# build zero state
zero_state = np.zeros(2**nqubits)
# initial params
params_len = len(symbols)
# fix numpy seed to ensure replicability of the experiment
seed = 10
np.random.seed(seed)
initial_params = np.random.uniform(-np.pi, np.pi, params_len)
print('Initial parameters:', initial_params)
# initial energy
c = substitute_ansatz_circuit(symbolic_circuit, initial_params)
initial_energy = unitary_expectation(h_qibo.matrix, c.get_unitary())
print('Target enegry:', target_energy)
print('Initial energy:', initial_energy)
print('Net difference:', initial_energy-target_energy)

In [ ]:
def vqe_loss(h_unitary, symbolic_circuit, params):
    c = substitute_ansatz_circuit(symbolic_circuit, params)
    return unitary_expectation(h_unitary, c.get_unitary())

objective = lambda params: vqe_loss(h_qibo.matrix, symbolic_circuit, params)

In [ ]:
max_iter = 1000
result = sp.optimize.minimize(
    objective,
    initial_params,
    method="cobyla",
    options={"disp": True, "maxiter": max_iter},
    tol=1e-2,
)

print(result.fun)
print(result.x)

In [ ]:
plt.plot(initial_params, label="initial params")
plt.plot(result.x, label='optimised params')
plt.legend()

In [ ]:
# save training results
import os
optimizer = 'cobyla'
nlayer = 2
nqubits = 10
folder_path = f'results/training_data/{optimizer}_{nqubits}q_{nlayer}l_XXZ'
os.makedirs(folder_path, exist_ok=True)
path_param = folder_path + f'/vqe_params.npy'
np.save(path_param, result.x)

## Save VQE qasm

In [ ]:
vqe_param_file = f'results/training_data/{optimizer}_{nqubits}q_{nlayer}l_XXZ/vqe_params.npy'
vqe_param = np.load(vqe_param_file)
vqe_circ = substitute_ansatz_circuit(symbolic_circuit, vqe_param)

In [ ]:
# save circuit
folder_path = f'../results/circuit_qasm/{optimizer}_{nqubits}q_{nlayer}l_XXZ/'
vqe_qasm = qasm.circuit_to_qasm(vqe_circ, folder_path + 'vqe_circ.qasm')

## Train DB-DOI

In [ ]:
vqe_circ_file = folder_path + 'vqe_circ.qasm'
vqe_circ = qasm.circuit_from_qasm(vqe_circ_file)
vqe_circ_inverse = vqe_circ.dagger()
U = vqe_circ.get_unitary()
U_dag = U.T.conjugate()
# check unitary
print('Check unitary (0):', np.linalg.norm(U@U_dag-np.eye(2**nqubits)))
# check expected energy
print('Expected energy:', unitary_expectation(h_qibo.matrix, U))

### Optimize D and s classically

In [ ]:
coef = [1] * nqubits
D_circ_qibo = hamiltonians.SymbolicHamiltonian(sum([coef[x]*Z(x) for x in range(nqubits)]))
s_space = np.linspace(0, 0.3, 10)
expect_ls = []
for s in s_space:
    d = D_circ_qibo.circuit(s).unitary()
    d_dag = d.conjugate()
    h = sp.linalg.expm(-1j*s*h_qibo.matrix)
    U1 = U @ d_dag @ U_dag @ h @ U @ d
    expect_ls.append(unitary_expectation(h_qibo.matrix, U1))

In [ ]:
plt.plot(s_space, expect_ls, label='DBQA')
plt.title(f'DBQA gain with time - XXZ (L={nqubits})')
min_expect = min(expect_ls)
min_idx = expect_ls.index(min_expect)
s_min = s_space[min_idx]
plt.scatter(s_min, min_expect, color='red', label=f'{round(s_min,2), round(min_expect,2)}')
plt.xlabel('time')
plt.ylabel(r'$\langle H\rangle$')
plt.legend()

In [ ]:
def magnetic_field(coefs, t):
    # implments e^{-itD}
    nqubits = len(coefs)
    qc = Circuit(nqubits)
    
    for idx, coef in enumerate(coefs):
        theta = 2 * t * coef / np.pi
        qc.Rz(theta, idx)
    return qc  

# define a loss function
def gci_loss(D_circ, H, U, s):
    """Return the analytical loss from a GCI step

    Args:
        D_circ (Pytket.Circuit): the magnetic field circuit in pytket
        H (np.array): the Input Hamiltonian matrix
        U (np.array): the VQE circuit unitary
        s (float): the GCI rotation duration
    """
    d = D_circ.get_unitary()
    d_dag = d.conjugate()
    h = sp.linalg.expm(-1j*s*H)
    U1 = U @ d_dag @ U_dag @ h @ U @ d
    return unitary_expectation(H, U1)

In [ ]:
objective = lambda D_s: gci_loss(magnetic_field(D_s[:-1], np.abs(D_s[-1])), h_qibo.matrix, U, np.abs(D_s[-1]))

In [ ]:
# setup initial D and s
s_init = s_min
D_s = np.append(coef, s_init)
print('Initial loss:', objective(D_s))

max_iter = 5000
result = sp.optimize.minimize(
    objective,
    D_s,
    method="COBYLA",
    options={"disp": True, "maxiter": max_iter},
    tol=1e-2,
)

print(result.fun)
print(result.x)

In [ ]:
if result.fun < objective(D_s):
    D_coef = result.x[:-1]
    s_min = result.x[-1]
    print('Use optimized params, s=', s_min)
else:
    D_coef = coef
    s_min = s_init

In [ ]:
folder_path = f'../results/training_data/{optimizer}_{nqubits}q_{nlayer}l_XXZ'
os.makedirs(folder_path, exist_ok=True)
path_param = folder_path + f'/doi_params.npy'
np.save(path_param, result.x)

In [ ]:
# XXZ model circuit (native gates)
def XX_interaction(qc, q0, q1, t):
    qc.H(q0), qc.H(q1)
    qc.ZZPhase(t*2/np.pi, q0, q1)
    qc.H(q0), qc.H(q1)

def YY_interaction(qc, q0, q1, t):
    qc.Sdg(q0), qc.Sdg(q1)
    XX_interaction(qc, q0, q1, t)
    qc.S(q0), qc.S(q1)

def ZZ_interaction(qc, q0, q1, t):
    qc.ZZPhase(t*2/np.pi, q0, q1)
    
def XXZ_decomposition(nqubits, t, delta=0.5, qc=None, layer=2, boundary='closed'):
    # This function generates the circuit that simulates e^{itH}
    # where H is the XXZ model
    # nqubits must be equal or greater than 2
    def generate_adjacent_pairs(n):
        if n < 2:
            raise ValueError("Input nqubits must be equal to or larger than 2.")
        even_starting_pairs = [[s, s+1] for s in range(0, n-1, 2)]
        odd_starting_pairs = [[s, s+1] for s in range(1,n-1, 2)]
        if boundary == 'periodic' or boundary == 'closed':
            # print('periodic case')
            if nqubits % 2 == 0:
                # even number of qubits add to odd pairs
                odd_starting_pairs.append([0, nqubits-1])
            else:
                even_starting_pairs.append([0, nqubits-1])
                # note that odd number of qubits result in degenerate ground states
        return even_starting_pairs, odd_starting_pairs
    even_starting_pairs, odd_starting_pairs = generate_adjacent_pairs(nqubits)
    if qc is None:
        qc = Circuit(nqubits) 
    if layer == 2:
        for q0, q1 in even_starting_pairs:
            XX_interaction(qc, q0, q1, t)
            YY_interaction(qc, q0, q1, t)
            ZZ_interaction(qc, q0, q1, t*delta)
        for q0, q1 in odd_starting_pairs:
            XX_interaction(qc, q0, q1, t)
            YY_interaction(qc, q0, q1, t)
            ZZ_interaction(qc, q0, q1, t*delta) 
    elif layer == 3:
        for q0, q1 in even_starting_pairs:
            XX_interaction(qc, q0, q1, t/2)
            YY_interaction(qc, q0, q1, t/2)
            ZZ_interaction(qc, q0, q1, t*delta/2)
        for q0, q1 in odd_starting_pairs:
            XX_interaction(qc, q0, q1, t)
            YY_interaction(qc, q0, q1, t)
            ZZ_interaction(qc, q0, q1, t*delta) 
        for q0, q1 in even_starting_pairs:
            XX_interaction(qc, q0, q1, t/2)
            YY_interaction(qc, q0, q1, t/2)
            ZZ_interaction(qc, q0, q1, t*delta/2)
    else:
        raise ValueError("Number of layers not supported, use either '2' or '3'.")
    return qc

In [ ]:
from copy import deepcopy
t = s_min
sequence = [i for i in range(nqubits)]
D_circ = magnetic_field(coef, t)
D_circ_inverse = D_circ.dagger()
qc = deepcopy(D_circ)
qc.add_circuit(vqe_circ, sequence)
qc.add_circuit(XXZ_decomposition(nqubits, t, layer=3, boundary='periodic'), sequence)
qc.add_circuit(vqe_circ_inverse, sequence)
qc.add_circuit(D_circ_inverse, sequence)
qc.add_circuit(vqe_circ, sequence)

In [ ]:
folder_path = f'../results/circuit_qasm/{optimizer}_{nqubits}q_{nlayer}l_XXZ/'
gci_c = qasm.circuit_to_qasm(qc, folder_path + 'vqe_gci_circ.qasm')